In [8]:
import sys
print(sys.executable)

c:\Users\bda\AppData\Local\Programs\Python\Python311\python.exe


In [9]:
import pyspark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('').getOrCreate()

In [10]:
import pyspark
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import*
from pyspark.sql.types import*
filepath = r'C:\Users\bda\Desktop\Fire CSV\sf-fire-calls.csv'

In [11]:
def create_sparkSession():
    spark=SparkSession.builder.appName('FireExample').getOrCreate()
    return spark


In [12]:
def create_dataframe(spark, filepath):
    df = spark.read.csv(filepath, header=True, inferSchema=True)
    df1=df.select('callType','callDate','City','Zipcode','Neighborhood','Delay')
    return df1

In [14]:
def clean_dataset(df):
    df1 = df.withColumn('Date', to_date(col('callDate'), 'MM/dd/yyyy')).drop('callDate')
    df2 = df1.withColumn('Year',year(col('Date')))\
    .withColumn('Month',month(col('Date')))\
    .withColumn('Week',weekofyear(col('Date')))
    return df2

In [15]:
spark = create_sparkSession()
df=create_dataframe(spark, filepath)
df = clean_dataset(df)
df.printSchema()

root
 |-- callType: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zipcode: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Delay: double (nullable = true)
 |-- Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Week: integer (nullable = true)



In [16]:
df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|        callType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|
|          Alarms|  SF|  94102|          Tenderloin|1.7833333|2002-01-11|2002|    1|   2|
|Medical I

In [18]:
#create user defined function
def mapSeason(data):
    if 2 < data < 6:
        return 'Spring'
    elif 5 < data < 9:
        return 'Summer'
    elif 8 < data < 12:
        return 'Autumn'
    else:
        return 'Winter'

seasonUDF = udf(mapSeason, StringType())
clean_df = df.withColumn('Season', seasonUDF(col('Month')))
clean_df.show()

+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|        callType|City|Zipcode|        Neighborhood|    Delay|      Date|Year|Month|Week|Season|
+----------------+----+-------+--------------------+---------+----------+----+-----+----+------+
|  Structure Fire|  SF|  94109|     Pacific Heights|     2.95|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94124|Bayview Hunters P...|      4.7|2002-01-11|2002|    1|   2|Winter|
|Medical Incident|  SF|  94102|          Tenderloin|2.4333334|2002-01-11|2002|    1|   2|Winter|
|    Vehicle Fire|  SF|  94110|      Bernal Heights|      1.5|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94109|    Western Addition|3.4833333|2002-01-11|2002|    1|   2|Winter|
|  Structure Fire|  SF|  94105|Financial Distric...|     1.75|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94112|Oceanview/Merced/...|2.7166667|2002-01-11|2002|    1|   2|Winter|
|          Alarms|  SF|  94102